# OLCF through the AmSC RIG

This notebook uses the **experimental** RIG helper in `amsc-client==0.7.1` to reach the OLCF S3M enclaves (`olcf-open`, `olcf-moderate`).

```text
AMSC_TOKEN (AmSC Keycard/PAT) -> RIG -> Credential Vault S3M token -> OLCF S3M IRI API v2
```

This is distinct from [`notebooks/olcf_facility_tutorial.ipynb`](../notebooks/olcf_facility_tutorial.ipynb), which calls S3M directly with your S3M token. Here the notebook only holds your AmSC PAT; RIG attaches the S3M token you saved in the Credential Vault. RIG still requires an active OLCF project allocation and a configured Credential Vault entry for each enclave. Never paste, print, decode, compare, serialize, or commit a token.

**IRI v2:** RIG advertises the OLCF mounts as IRI **v2**, but `client.facility_via_rig()` in `amsc-client==0.7.1` only accepts v1. The repository helper `olcf_s3m.olcf_via_rig()` (`notebooks/olcf_s3m.py`) bridges this for the OLCF mounts only, and Step 2 verifies that requests go to `/api/v2`.

**Validation status (2026-10-09):** this notebook is **statically validated** against the published wheel.
- **Live-validated on 2026-10-09 through staging RIG on both `olcf-open` and `olcf-moderate`:** discovery (IRI v2), the `/api/v2` check, the protected `whoami` read, resource discovery, and the optional job submission with a bounded wait (a Defiant job and a Frontier job each completed with exit code 0).
- Job update and cancel are not covered by this notebook and are **not live-validated**.
- Submission is disabled by default.

This example discovers the OLCF enclaves, checks that the client calls v2, proves the routed credential with a protected `whoami` read, inspects compute resources, and provides an optional bounded job workflow.

## Before you run: save your S3M token in the Credential Vault

`AMSC_TOKEN` in this notebook must remain your **AmSC PAT**. OLCF is a Tier-3 (vaulted) facility: RIG uses an S3M token that *you* save in the Credential Vault. Do not replace `AMSC_TOKEN` with an S3M token.

1. **Get an S3M Project Access Token** for each enclave you will use (open and moderate are separate enclaves with separate tokens). On the ORNL S3M token page, choose the **AmSC IRI API** permission (maximum lifetime 1 week + 8 hours). See [S3M token management](https://docs.olcf.ornl.gov/services_and_applications/s3m/overview.html#token-management).

2. Open the [staging RIG Credential Vault](https://rig-ui.staging.american-science-cloud.org/credential-vault) and sign in with your AmSC identity.

3. Confirm that the **active project** shown by the Credential Vault is the same project encoded in the AmSC PAT used by this notebook. Vaulted credentials are project-scoped.

4. On the **OLCF Open** (or **OLCF Moderate**) card, paste the S3M token into **Bearer token** and click **Save**. The card should change from **Needs setup** to **Ready**.

5. Click **Verify via RIG**. Do not continue until it succeeds; a saved credential alone does not prove the routed request works.

6. When the S3M token expires, or the protected read reports "No facility credential configured", create a new S3M token and **Rotate** the saved credential, then **Verify via RIG** again.

Never place an S3M token in notebook cells, chat, shell history, screenshots, or committed files. It belongs only in the Credential Vault form.

In [ ]:
import os
import sys
from importlib.metadata import version
from pathlib import Path

from amsc_client import Client
from amsc_client.core.exceptions import ApiError, AuthenticationError

# OLCF S3M helper (notebooks/olcf_s3m.py): bridges RIG's IRI v2 OLCF mounts
for candidate in (Path.cwd().parent / "notebooks", Path.cwd() / "notebooks"):
    if (candidate / "olcf_s3m.py").exists():
        sys.path.insert(0, str(candidate))
        break
from olcf_s3m import RigAccessError, olcf_via_rig, recent_requests, rig_whoami, verify_api_version

STAGING_API = "https://api.staging.american-science-cloud.org/api/current"
STAGING_RIG = "https://rig.staging.american-science-cloud.org"
AMSC_TOKEN = os.environ["AMSC_TOKEN"]
assert version("amsc-client") == "0.7.1", "Install requirements.txt and restart this kernel"
client = Client(token=AMSC_TOKEN, base_url=STAGING_API)
print("Client release:", version("amsc-client"))

## Step 1: Discover the OLCF enclaves
`rig_facilities()` uses the AmSC Keycard/PAT to read RIG's facility inventory. OLCF appears as two Tier-3 facilities, `olcf-open` and `olcf-moderate`, each advertising IRI `v2` and the probe path `/account/whoami`. Set `OLCF_ENCLAVE` (or edit `ACTIVE_ENCLAVE`) to choose one.

In [ ]:
ACTIVE_ENCLAVE = os.getenv("OLCF_ENCLAVE", "open")   # "open" or "moderate"
FACILITY_NAME = f"olcf-{ACTIVE_ENCLAVE}"

facilities = client.rig_facilities(rig_url=STAGING_RIG)
olcf_entries = [f for f in facilities if f.name.startswith("olcf-")]
for f in olcf_entries:
    print(f.name, f.tier, f.api_version, f.probe_path)

olcf_meta = next((f for f in olcf_entries if f.name == FACILITY_NAME), None)
if olcf_meta is None:
    raise RuntimeError(f"{FACILITY_NAME} is not advertised by this RIG deployment")
if olcf_meta.api_version.strip("/") != "v2":
    raise RuntimeError(f"Expected IRI v2 for {FACILITY_NAME}; RIG advertises {olcf_meta.api_version!r}")

## Step 2: Connect and confirm the client uses API v2
`olcf_via_rig()` returns an amsc-client facility routed through `<rig>/rig/external/olcf-<enclave>`. `verify_api_version()` builds (without sending) the URL the client would call and stops with an error unless it is under `/api/v2`.

In [ ]:
olcf = olcf_via_rig(client, ACTIVE_ENCLAVE, rig_url=STAGING_RIG)
verify_api_version(olcf, token=AMSC_TOKEN)

## Step 3: Protected read — `whoami` through RIG
S3M has no Account API (`projects()` is refused), so the protected read for OLCF is RIG's probe path, `GET /api/v2/account/whoami`. Success proves the whole chain: your AmSC PAT was accepted by RIG, RIG found your vaulted S3M token, and S3M accepted it. `username` is the account jobs run as (your project's automation user, `prj###_auser`).

- **"No facility credential configured"**: the Credential Vault has no S3M token for this enclave and project; see the setup section.
- **`S3M code 16` / unauthorized**: S3M rejected the vaulted token (expired, revoked, or issued for the other enclave); **Rotate** it.

In [ ]:
try:
    identity = rig_whoami(AMSC_TOKEN, ACTIVE_ENCLAVE, rig_url=STAGING_RIG)
except RigAccessError as exc:
    if "No facility credential configured" in exc.detail:
        raise RuntimeError(
            f"Save an S3M token on the OLCF {ACTIVE_ENCLAVE.title()} card in the Credential Vault, "
            f"then click Verify via RIG. ({exc})"
        ) from None
    raise

print(f"Protected whoami succeeded through RIG ({FACILITY_NAME}):")
for key, value in identity.items():
    if not key.startswith("_"):
        print(f"  {key:12s} {value}")

## Step 4: Resource discovery
Set `RESOURCE_NAME` to one of the listed names (case-insensitive): `"Defiant"`, `"Wombat"`, `"Quokka"` or `"Odo"` on open, `"Frontier"` on moderate. `None` uses the enclave default. The request list shows that discovery went through the RIG mount under `/api/v2`.

In [ ]:
resources = olcf.resources()
for resource in resources:
    print(resource.name, getattr(resource.resource_type, "value", resource.resource_type), resource.status)

print("\nRequests sent through RIG:")
for request in recent_requests(10, base_url=olcf.base_url):
    print(" ", request)

RESOURCE_NAME = None        # e.g. "Odo"; None = enclave default
DEFAULT_RESOURCE = {"open": "Defiant", "moderate": "Frontier"}
compute = olcf.resource(RESOURCE_NAME or DEFAULT_RESOURCE[ACTIVE_ENCLAVE])
print("\nSelected:", compute.name, compute.status)

## Optional job submission (default off)
Set `OLCF_ACCOUNT`, `OLCF_QUEUE` and `OLCF_WORK_DIR` before enabling the gate. The working directory must exist and be writable by your project's automation user; run-unique names prevent collisions. Enabling this cell creates a real scheduler job.

In [ ]:
from datetime import datetime, timezone

SUBMIT_JOB = False
OLCF_ACCOUNT = os.getenv("OLCF_ACCOUNT", "")
OLCF_QUEUE = os.getenv("OLCF_QUEUE", "")
OLCF_WORK_DIR = os.getenv("OLCF_WORK_DIR", "")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
JOB_NAME = f"amsc-rig-{FACILITY_NAME}-{RUN_ID}"

if SUBMIT_JOB:
    missing = [name for name, value in {"OLCF_ACCOUNT": OLCF_ACCOUNT, "OLCF_QUEUE": OLCF_QUEUE, "OLCF_WORK_DIR": OLCF_WORK_DIR}.items() if not value]
    if missing:
        raise RuntimeError("Missing required environment settings: " + ", ".join(missing))
    stdout_path = f"{OLCF_WORK_DIR}/{JOB_NAME}.stdout"
    stderr_path = f"{OLCF_WORK_DIR}/{JOB_NAME}.stderr"
    job = compute.submit(
        executable="/bin/bash", arguments=["-lc", f"echo 'Hello from AmSC through RIG' {RUN_ID}; hostname; sleep 60"],
        directory=OLCF_WORK_DIR, name=JOB_NAME, queue=OLCF_QUEUE, account=OLCF_ACCOUNT,
        duration=300, nodes=1, stdout_path=stdout_path, stderr_path=stderr_path,
    )
    print("Submitted", JOB_NAME, "as job", job.id)
else:
    print("SUBMIT_JOB=False: no scheduler job was submitted.")

## Bounded wait
This executes only if the default-off submission gate was enabled and submission succeeded. S3M does not serve the Filesystem API yet, so read the job's output files on the OLCF filesystem directly.

In [ ]:
if SUBMIT_JOB:
    completed = job.wait(timeout=600, poll_interval=10)
    print("Final state:", completed.state, "exit code:", completed.exit_code)
    print("Output files on the OLCF filesystem:", stdout_path, stderr_path)